In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import ast
import math
import itertools
import time
from tqdm import tqdm
from tqdm.contrib.itertools import product
import numpy as np

In [3]:
engine_size = pd.read_csv('/content/drive/MyDrive/Gaming note/GearCity/engine_size.csv')
Layouts = pd.read_csv('/content/drive/MyDrive/Gaming note/GearCity/Engine_Layouts.csv')
Cylinders = pd.read_csv('/content/drive/MyDrive/Gaming note/GearCity/Engine_Cylinders.csv')
Fuel = pd.read_csv('/content/drive/MyDrive/Gaming note/GearCity/Engine_Fuel.csv')
Valvetrain = pd.read_csv('/content/drive/MyDrive/Gaming note/GearCity/Engine_Valvetrain.csv')
Induction = pd.read_csv('/content/drive/MyDrive/Gaming note/GearCity/Engine_Induction.csv')
world_events = pd.read_csv('/content/drive/MyDrive/Gaming note/GearCity/world_event.csv')

In [4]:
# Function to convert string representation of list back to list
def str_to_list(s):
    return ast.literal_eval(s)

# Apply the function to the columns
Layouts["Cylinders"] = Layouts["Cylinders"].apply(str_to_list)
Layouts["Fuel Types"] = Layouts["Fuel Types"].apply(str_to_list)
Layouts["Inductions"] = Layouts["Inductions"].apply(str_to_list)

In [5]:
# Fix value
designRandomVal = 1
Marq_DesignEngineSkill = 100

In [6]:
def cal_optimal_performance(Bore_mm,Stroke_mm):
  Displacement_CC = (0.7854 * ((Bore_mm/10) * (Bore_mm/10)) * (Stroke_mm/10) * SubComponent_Cylinders_CylinderCount)
  #Torque
  Torque = 10 + (Marq_DesignEngineSkill/20.0) + \
    ((((25) * ((Slider_Performance_Torque - 0.4)*1.5)*ex_1d01p_year99) + \
    ((4*(SubComponent_Layout_Length + SubComponent_Layout_Width) )*ex_1d005p_year99) - \
    (14 * (Slider_Performance_FuelEconomy+Slider_DesignFocus_FuelEconomy) * ex_1d004p_year99) +\
    (SubComponent_Layout_PowerRatings*5 + SubComponent_Cylinders_PowerRating*13 +\
    SubComponent_FuelType_PowerRating*24 + 100*SubComponent_Induction_PowerRating +\
    (5 * ex_1d004p_year99 * Slider_DesignFocus_Performance) +\
    8*(Slider_Technology_Components+Slider_Technology_Materials + \
    Slider_Technology_Technologies +Slider_Technology_Techniques))*ex_1d0024p_year99))

  Torque = Torque * ((SubComponent_Cylinders_CylinderCount * Stroke_mm*0.93 * Bore_mm*0.9)*0.000027)+5

  if year < 2050:
      Torque = Torque * ex_0d996p_year50R

  Torque = Torque * SubComponent_Valve_PowerRating

  #RPM
  tmpAY = AdjustedYear
  if tmpAY > 80:
      tmpAY = 80 + ((AdjustedYear-80)/5)


  rpm = ((((tmpAY**4)*0.00000420875) - \
      ((19*(tmpAY**3))*0.00016835) + ((427*(tmpAY**2))*0.00126) +\
      ((1315*(tmpAY))*0.01515) + 620 ) + (265 * ex_1d01p_year99 * Slider_DesignFocus_Performance) +\
      (465 * ex_1d0105p_year99 * (Slider_Performance_Revolutions*5.5)) -\
      (10 * ex_1d01p_year99 * SubComponent_Induction_PowerRating) +\
      (55 * ex_1d005p_year99 * (1-Slider_Layout_Weight)) - (30* ex_1d005p_year99 *\
      (Slider_DesignFocus_FuelEconomy + Slider_Performance_FuelEconomy))+  \
      (25 * ex_1d01p_year99 * Slider_Technology_Components) + \
      (25 * ex_1d01p_year99 * Slider_Technology_Materials) + \
      (25 * ex_1d01p_year99 * Slider_Technology_Technologies)) * SubComponent_FuelType_RPM


  rpm = rpm * SubComponent_Valve_RPM

  rpm = rpm - ((rpm/1.5) * (Stroke_mm/221.136364))

  if rpm < 25:
      rpm = 25
  hp = (Torque * rpm) / 5252
  # Unit cost
  Unit_Costs =((((((70* ex_1d01p_year99 * (((1-Slider_Layout_Length) + (1-Slider_Layout_Width))/2.0)) +\
    (220 * ex_1d004p_year99 * (((0.25+(Slider_Performance_Revolutions * \
    Slider_Performance_Revolutions) + (Slider_Performance_Torque * Slider_Performance_Torque))/2.0) -\
    (0.5-(Slider_Performance_FuelEconomy*Slider_Performance_FuelEconomy)) )  ) +\
    (60 * ex_1d01p_year99) *  ((Slider_Performance_Revolutions * Slider_Performance_Revolutions) +\
    (Slider_Performance_Torque * Slider_Performance_Torque)) +\
    220 * ex_1d008p_year99*(0.1+(((Slider_Technology_Materials*Slider_Technology_Materials)+\
    (Slider_Technology_Techniques*Slider_Technology_Techniques ) + \
    (Slider_Technology_Components*Slider_Technology_Components))) ) +\
    170 * ex_1d008p_year99*(Slider_Technology_Technologies*Slider_Technology_Technologies) +\
    50 * ex_1d0035p_year99 * (Slider_DesignFocus_Dependability * Slider_DesignFocus_Dependability) +\
    180 * ex_1d0035p_year99 * (Slider_DesignFocus_Performance * Slider_DesignFocus_Performance )+\
    (260 * ex_1d006p_year99 * (2.168 * Slider_Layout_Displacement**1.5 -4.44 * Slider_Layout_Displacement**3 +\
    2.646  * Slider_Layout_Displacement**4.5 + 3.126 * Slider_Layout_Displacement**6 )+\
    (70 * ex_1d005p_year99 * (SubComponent_Cylinders_CylinderCount/6.0) + \
    (.75 +  Slider_Layout_Displacement**1.5) - (Slider_Layout_Weight**2)) + 10 * \
    (Slider_DesignFocus_FuelEconomy**2) - 50 ) *  ex_1d003p_year99 + \
    (160 * SubComponent_Cylinders_UnitCosts)**ex_1d003p_year99 +\
    (120 * SubComponent_Layout_UnitCosts)**ex_1d004p_year99 +\
    (140 * SubComponent_Valve_UnitCosts)**ex_1d004p_year99 +\
    (435 * SubComponent_Induction_UnitCosts)**ex_1d004p_year99 +\
    (120 * SubComponent_FuelType_UnitCosts)**ex_1d004p_year99) * \
    (.125 + 0.12 * SubComponent_Cylinders_CylinderCount)) * \
    (Global_Interest_Rate/2.0)) + 50)  * carPriceRate) * (designRandomVal)

  Hyper_Sliders = ((Slider_Layout_Displacement*2 + (1-Slider_Layout_Length) + \
      (1-Slider_Layout_Width) + (1-Slider_Layout_Weight)) +\
      (Slider_Performance_Revolutions + Slider_Performance_Torque + Slider_Performance_FuelEconomy) +\
      (Slider_DesignFocus_Performance + Slider_DesignFocus_FuelEconomy  + Slider_DesignFocus_Dependability) +\
      (Slider_Technology_Materials + Slider_Technology_Components + Slider_Technology_Techniques +\
      Slider_Technology_Technologies))/13.0

  Hyper_Costs = 475 * ex_1d04p_year99 * (Hyper_Sliders*Hyper_Sliders*Hyper_Sliders*Hyper_Sliders)


  Unit_Costs = Unit_Costs + Hyper_Costs - ((Unit_Costs/10) * (Marq_DesignEngineSkill/100))

  return Displacement_CC,Torque*1.356,hp,Unit_Costs

In [7]:
# Setup
year = 1932
max_cost = 300
max_cc = 6000
# ['Electric', 'Flat', 'H', 'I', 'Radial', 'Rotary', 'Single', 'Steam', 'U', 'V', 'VV', 'W', 'Wankel', 'X']
test_engine = ['Flat', 'H', 'I', 'Radial', 'Rotary', 'Single', 'U', 'V', 'VV', 'W', 'Wankel', 'X']
# ['Autogas', 'Diesel', 'E85', 'Electric I', 'Electric II', 'Electric III', 'Electric IV', 'Electric V',
# 'Gasoline', 'Hybrid', 'Hydrogen', 'Natural Gas', 'Water']
test_fuel = ['Gasoline']

# Cost impact
Slider_Layout_Length = 50/100
Slider_Layout_Width = 50/100
Slider_DesignFocus_Dependability = 50/100
# Performance impact
Slider_Layout_Weight = 50/100
Slider_Performance_FuelEconomy = 30/100
Slider_Technology_Components = 0/100
Slider_Technology_Materials = 0/100
Slider_Technology_Technologies = 0/100
Slider_Technology_Techniques = 0/100

In [8]:
# Calculate year factor
if year > 2020:
  ex_0d996p_year50R = 0.901037361
else:
  ex_0d996p_year50R = 0.996**(2050-year)

AdjustedYear = year - 1899
ex_1d0024p_year99 = 1.0024 **(year-1899)
ex_1d0035p_year99 = 1.0035 **(year-1899)
ex_1d005p_year99 = 1.005 **(year-1899)
ex_1d006p_year99 = 1.006 **(year-1899)
ex_1d008p_year99 = 1.008 **(year-1899)
ex_1d025p_year99 = 1.025 **(year-1899)
ex_1d033p_year99 = 1.033 **(year-1899)
ex_1d038p_year99 = 1.038 **(year-1899)
ex_1d04p_year99 = 1.04 **(year-1899)
ex_1d0023p_year99 = 1.0023 **(year-1899)
ex_1d003p_year99 = 1.003 **(year-1899)
ex_1d004p_year99 = 1.004 **(year-1899)
ex_1d0051p_year99 = 1.0051 **(year-1899)
ex_1d007p_year99 = 1.007 **(year-1899)
ex_1d01p_year99 = 1.01 **(year-1899)
ex_1d03p_year99 = 1.03 **(year-1899)
ex_1d035p_year99 = 1.035 **(year-1899)
ex_1d039p_year99 = 1.039 **(year-1899)
ex_1d05p_year99 = 1.05 **(year-1899)
ex_1d0105p_year99 = 1.0105 **(year-1899)

Global_Interest_Rate = world_events[world_events['year']==math.floor(year)]['interest_rate'].item()
carPriceRate = world_events[world_events['year']==math.floor(year)]['carprice_rate'].item()

In [10]:
max_cal_hp = 0
optimal_setting = []
for lay_index, lay_row in Layouts.iterrows():
  if lay_row['Name'] in test_engine and lay_row["Year"] <= year:
    # Engine work with Cylinders / Fuel Types / Inductions / Valve
    select_cylinders = lay_row["Cylinders"]
    select_fuel = lay_row["Fuel Types"]
    select_inductions = lay_row["Inductions"]
    if lay_row["Valve"] == 1:
      select_valve = ['No Valve']
    elif lay_row["Valve"] == 2:
      select_valve = ['Poppet Valve','Sleeve Valve']
    elif lay_row["Valve"] == 3:
      select_valve = ['F Head','L Head','OHV','SOHC','T Head','Two Stroke']
      if year >= 1904:
        select_valve.append('DOHC')
    # Engine layout detail for calculation
    SubComponent_Layout_Length = lay_row['Length']
    SubComponent_Layout_Width = lay_row['Width']
    SubComponent_Layout_PowerRatings = lay_row['Power']
    SubComponent_Layout_UnitCosts = lay_row['Costs']
    if year%5 == 0 or year > 2020:
      if year > 2020:
        limit_year = 2020
      limit_year = year//5*5
      min_bore = engine_size[(engine_size['Name']==lay_row['Name'])&(engine_size['Year']==limit_year)]['Min_Bore'].item()
      max_bore = engine_size[(engine_size['Name']==lay_row['Name'])&(engine_size['Year']==limit_year)]['Max_Bore'].item()
      min_stroke = engine_size[(engine_size['Name']==lay_row['Name'])&(engine_size['Year']==limit_year)]['Min_Stroke'].item()
      max_stroke = engine_size[(engine_size['Name']==lay_row['Name'])&(engine_size['Year']==limit_year)]['Max_Stroke'].item()
    else:
      limit_year_min = year//5*5
      limit_year_max = (year//5+1)*5
      min_bore_y_min = engine_size[(engine_size['Name']==lay_row['Name'])&(engine_size['Year']==limit_year_min)]['Min_Bore'].item()
      max_bore_y_min = engine_size[(engine_size['Name']==lay_row['Name'])&(engine_size['Year']==limit_year_min)]['Max_Bore'].item()
      min_stroke_y_min = engine_size[(engine_size['Name']==lay_row['Name'])&(engine_size['Year']==limit_year_min)]['Min_Stroke'].item()
      max_stroke_y_min = engine_size[(engine_size['Name']==lay_row['Name'])&(engine_size['Year']==limit_year_min)]['Max_Stroke'].item()
      min_bore_y_max = engine_size[(engine_size['Name']==lay_row['Name'])&(engine_size['Year']==limit_year_max)]['Min_Bore'].item()
      max_bore_y_max = engine_size[(engine_size['Name']==lay_row['Name'])&(engine_size['Year']==limit_year_max)]['Max_Bore'].item()
      min_stroke_y_max = engine_size[(engine_size['Name']==lay_row['Name'])&(engine_size['Year']==limit_year_max)]['Min_Stroke'].item()
      max_stroke_y_max = engine_size[(engine_size['Name']==lay_row['Name'])&(engine_size['Year']==limit_year_max)]['Max_Stroke'].item()
      min_bore = min_bore_y_max + ((min_bore_y_min-min_bore_y_max)*(5-(year%5))*0.2)
      max_bore = max_bore_y_max + ((min_bore_y_min-max_bore_y_max)*(5-(year%5))*0.2)
      min_stroke = min_stroke_y_max + ((min_stroke_y_min-min_stroke_y_max)*(5-(year%5))*0.2)
      max_stroke = max_stroke_y_max + ((max_stroke_y_min-max_stroke_y_max)*(5-(year%5))*0.2)

    for fuel,cylinder,induct,valve in product(select_fuel,select_cylinders,select_inductions,select_valve):
      if fuel in test_fuel and Fuel[Fuel['Name']==fuel]['Year'].item() <= year-1 and \
      Cylinders[Cylinders['Name']==cylinder]['Year'].item() <= year-1 and \
      Induction[Induction['Name']==induct]['Year'].item() <= year-1:
        #Fuel
        SubComponent_FuelType_RPM = Fuel[Fuel['Name']==fuel]['RPM'].item()
        SubComponent_FuelType_PowerRating = Fuel[Fuel['Name']==fuel]['Power'].item()
        SubComponent_FuelType_UnitCosts = Fuel[Fuel['Name']==fuel]['Cost'].item()
        #Cylinders
        SubComponent_Cylinders_PowerRating = Cylinders[Cylinders['Name']==cylinder]['Power'].item()
        SubComponent_Cylinders_CylinderCount = Cylinders[Cylinders['Name']==cylinder]['Number of Cylinders'].item()
        SubComponent_Cylinders_UnitCosts = Cylinders[Cylinders['Name']==cylinder]['Cost'].item()
        #Induction
        SubComponent_Induction_PowerRating = Induction[Induction['Name']==induct]['Power'].item()
        SubComponent_Induction_UnitCosts = Induction[Induction['Name']==induct]['Cost'].item()
        #Valvetrain
        SubComponent_Valve_RPM = Valvetrain[Valvetrain['Name']==valve]['RPM'].item()
        SubComponent_Valve_PowerRating = Valvetrain[Valvetrain['Name']==valve]['Power'].item()
        SubComponent_Valve_UnitCosts = Valvetrain[Valvetrain['Name']==valve]['Costs'].item()
        for perf_torque,perf_revo,design_fuel,design_perf in \
        itertools.product(range(30,101,5),range(30,101,5),
                          range(0,101,5),range(0,101,5)):
          Slider_Performance_Torque = perf_torque/100
          Slider_Performance_Revolutions = perf_revo/100
          Slider_DesignFocus_FuelEconomy = design_fuel/100
          Slider_DesignFocus_Performance = design_perf/100
          for check_bore,check_stroke in itertools.product(range(int((min_bore//1+1)*100),int(max_bore//1*100),25),range(int((min_stroke//1+1)*100),int(max_stroke//1*100),25)):
            Slider_Layout_Displacement = ((check_bore-min_bore)/(max_bore-min_bore)+(check_stroke-min_stroke)/(max_stroke-min_stroke))/2
            cal_cc = (0.7854 * ((check_bore/10) * (check_stroke/10)) * (check_stroke/10) * SubComponent_Cylinders_CylinderCount)
            if cal_cc < max_cc:
              continue
            cal_cc,cal_torque,cal_hp,cal_cost = cal_optimal_performance(check_bore,check_stroke)
            if cal_cost < max_cost:
              continue
            if max_cal_hp < cal_hp:
              max_cal_hp = cal_hp
              optimal_setting = [fuel,cylinder,induct,valve,perf_torque,perf_revo,design_fuel,design_perf,check_bore,check_stroke]

  0%|          | 0/1440 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
optimal_setting